# HAYI — minimal demo

One class, `Agent`, is the whole API. It resolves its own configuration from
`AGENT_*` environment variables (or constructor overrides), renders a prompt
template the consumer owns, runs the agentic loop and streams every block to
the console.

Setup:

```
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python -e .
export AGENT_API_KEY=...        # never commit it
```

Everything else (`AGENT_MODEL`, vision model, repository, ...) is optional
configuration with sensible defaults.

In [ ]:
# Some input to summarise. `input=` accepts a file path or raw text; when it
# is a path the file is read and its text becomes `config.input`.
from pathlib import Path

Path("notes.md").write_text(
    """
# Meeting notes

- The agent harness exposes a single `Agent` class.
- Sessions remember their conversation between runs.
- Every tool call is streamed with its arguments and outcome.
"""
)
print(Path("notes.md").read_text())

In [ ]:
# Run the agent. It resolves its configuration from AGENT_* environment
# variables (or constructor overrides), renders the template — `config.input`
# is the text of notes.md — and streams every block to the console.
#
# This needs AGENT_API_KEY (or AGENT_BASE_URL pointing at an OpenAI-compatible
# endpoint, e.g. a local one) to be set in the kernel environment.
import asyncio
from agent import Agent

TEMPLATE = """
Summarise the following notes as bullet points.

{{ config.input }}
"""

result = asyncio.run(Agent().run(TEMPLATE, input="notes.md"))
print("done:", "ok" if result.ok else result.error)


In [ ]:
# The run was streamed to the console above; the returned `RunResult` carries
# the same content for programmatic use.
print("output:", result.output)
print("session:", result.session_id)
print("tool calls:", [call.signature() for call in result.tools])
print("error:", result.error)

In [ ]:
# The session remembers its conversation: passing the same session id runs
# with the earlier turns replayed, so the model knows what it did before.
followup = asyncio.run(
    Agent().run(
        """
How many bullet points did you produce, and what was the first one?

{{ config.input }}
""",
        input="notes.md",
        session=result.session_id,
    )
)
print("output:", followup.output)
print("session:", followup.session_id, "(same as before)" if followup.session_id == result.session_id else "(new!)")


The session remembers its conversation, so running again with the same session
id continues it. Nothing else is required — see `README.md` and `GUIDE.md` for
embedding (`Engine`), the web UI (`python agent.py --serve`) and the full
configuration reference.